In [1]:
from google.colab import drive
import os
import shutil

drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/LoRA_LLM'

for sub in ('src', 'data', 'output'):
    os.makedirs(os.path.join(PROJECT_DIR, sub), exist_ok=True)

# Битый кэш модели на Диске удаляем (после первой очистки строку можно оставить: она безвредна).
shutil.rmtree(os.path.join(PROJECT_DIR, 'hf_cache'), ignore_errors=True)

# Кэш Hugging Face должен лежать на локальном диске Colab, а не на Google Диске.
os.environ.pop('HF_HOME', None)

%cd /content/drive/MyDrive/LoRA_LLM/src

Mounted at /content/drive
/content/drive/MyDrive/LoRA_LLM/src


In [2]:
from google.colab import files

uploaded = files.upload()
print(list(uploaded))

Saving requirements.txt to requirements.txt
['requirements.txt']


In [3]:
import os

# torchao 0.10.0 из Colab несовместим с новым peft, а нам он не нужен (квантизации нет).
!pip uninstall -y torchao

if os.path.exists('../requirements.txt'):
    !pip install -q -r ../requirements.txt
else:
    !pip install -q -U transformers peft accelerate

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [4]:
import torch
print("CUDA доступна:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("bf16:", torch.cuda.is_bf16_supported())
else:
    print("GPU нет: Runtime → Change runtime type → T4 GPU")
!nvidia-smi

CUDA доступна: True
GPU: Tesla T4
bf16: True
Sun Nov  8 14:20:40 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P0             26W /   70W |     107MiB /  15360MiB |      3%      Default |
|                                         |                        |                  N/A |
+--

## 1. prepare_data.py

In [5]:
%%writefile prepare_data.py
"""Скачать страницы lib.ru и нарезать их на рассказы в JSONL."""

from __future__ import annotations

import argparse
import json
import re
import subprocess
from pathlib import Path

SOURCES = {
    "harms": ["https://www.lib.ru/HARMS/harms.txt"],
    # Тома 1–10 полного собрания. text_0032 — подписи к рисункам, не проза.
    "chekhov": [
        f"http://az.lib.ru/c/chehow_a_p/text_{number:04d}.shtml"
        for number in (10, 20, 30, 40, 50, 60, 70, 80, 90, 100)
    ],
}

# Хармс на lib.ru — KOI8-R, тома Чехова на az.lib.ru — windows-1251.
ENCODINGS = {"harms": "koi8-r", "chekhov": "windows-1251"}

H2_RE = re.compile(r"<h2>(.*?)</h2>", re.IGNORECASE | re.DOTALL)
ANCHOR_RE = re.compile(r'<a name="(\d+)"></a>', re.IGNORECASE)
TITLE_RE = re.compile(r"<FONT color=darkred><B>(.*?)</B>", re.IGNORECASE | re.DOTALL)
H4_TITLE_RE = re.compile(r"<h4>.*?<b>(.*?)</b>", re.IGNORECASE | re.DOTALL)
TAG_RE = re.compile(r"<[^>]+>")
MIN_CHARS = 200


def download(url: str, encoding: str) -> str:
    # curl -4: у lib.ru есть AAAA, а маршрута IPv6 часто нет.
    completed = subprocess.run(
        ["curl", "-4", "-fsSL", "--retry", "3", "--max-time", "180", url],
        check=True,
        capture_output=True,
    )
    if not encoding:
        raise NotImplementedError("укажите кодировку страницы в ENCODINGS")
    return completed.stdout.decode(encoding, errors="replace")


def html_to_text(fragment: str) -> str:
    text = TAG_RE.sub("", fragment)
    text = text.replace("&quot;", '"').replace("&lt;", "<").replace("&gt;", ">")
    text = text.replace("\xa0", " ")
    # склейка переносов: «по-\nехал» -> «поехал»
    text = re.sub(r"(?<=[а-яёА-ЯЁ])-[ \t]*\n[ \t]*(?=[а-яёА-ЯЁ])", "", text)
    text = re.sub(r"[ \t]*\n[ \t]*", " ", text)
    text = re.sub(r"[ ]{2,}", " ", text)
    return text.strip()


def keep_story(title: str, text: str) -> bool:
    """Оставляем только сами рассказы: без оглавления, примечаний, писем и т.п."""
    if len(text) < MIN_CHARS:
        return False
    lowered = title.casefold()
    not_story = (
        "оглавление",
        "содержание",
        "от редакции",
        "от издателя",
        "предисловие",
        "примечани",
        "комментари",
        "письма",
        "титул",
        "список",
    )
    return not any(word in lowered for word in not_story)


def split_by_h2(page: str) -> list[dict]:
    """Куски до первого <h2> отбрасываются. Так нарезана страница Хармса."""
    parts = H2_RE.split(page)
    stories = []
    for title_html, body_html in zip(parts[1::2], parts[2::2]):
        title = html_to_text(title_html)
        text = html_to_text(body_html)
        if keep_story(title, text):
            stories.append({"title": title, "text": text})
    return stories


def split_by_anchor(page: str) -> list[dict]:
    """Тома Чехова: каждый рассказ начинается с <a name="N"> и заголовка."""
    parts = ANCHOR_RE.split(page)
    stories = []
    for body_html in parts[2::2]:
        title_match = TITLE_RE.search(body_html) or H4_TITLE_RE.search(body_html)
        if title_match is None:
            continue
        title = html_to_text(title_match.group(1))
        text = html_to_text(body_html)
        if keep_story(title, text):
            stories.append({"title": title, "text": text})
    return stories


def split_stories(page: str) -> list[dict]:
    by_heading = split_by_h2(page)
    by_anchor = split_by_anchor(page)
    if len(by_anchor) >= len(by_heading) and by_anchor:
        return by_anchor
    return by_heading


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--author", choices=sorted(SOURCES), required=True)
    parser.add_argument("--out", type=Path, required=True)
    args = parser.parse_args()

    stories = []
    seen_titles = set()
    for url in SOURCES[args.author]:
        print(f"качаю {url}", flush=True)
        page = download(url, ENCODINGS[args.author])
        for story in split_stories(page):
            key = story["title"].casefold()
            if key in seen_titles:
                continue
            seen_titles.add(key)
            stories.append(story)
    if not stories:
        raise SystemExit("после чистки не осталось рассказов, проверьте кодировку и нарезку")

    args.out.parent.mkdir(parents=True, exist_ok=True)
    with args.out.open("w", encoding="utf-8") as fh:
        for story in stories:
            fh.write(json.dumps(story, ensure_ascii=False) + "\n")
    chars = sum(len(s["text"]) for s in stories)
    print(f"{args.author}: {len(stories)} рассказов, {chars} символов -> {args.out}")


if __name__ == "__main__":
    main()

Overwriting prepare_data.py


## 2. train_lora.py

In [6]:
%%writefile train_lora.py
"""Обучение LoRA на рассказах одного автора."""

from __future__ import annotations

import os

# httpx не принимает схему socks:// — заменяем на socks5:// до импорта huggingface.
for _key, _value in list(os.environ.items()):
    if _key.lower().endswith("proxy") and isinstance(_value, str) and _value.startswith("socks://"):
        os.environ[_key] = "socks5://" + _value[len("socks://") :]

import argparse
import json
from pathlib import Path

import torch
from peft import LoraConfig, TaskType, get_peft_model
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Trainer,
    TrainingArguments,
)


def pick_dtype() -> torch.dtype:
    if torch.cuda.is_available():
        return torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    return torch.float32


def make_messages(story: str) -> list[dict]:
    """Диалог для обучения: одна реплика user, без system (как в generate.py)."""
    return [
        {"role": "user", "content": "Напиши короткий рассказ."},
        {"role": "assistant", "content": story},
    ]


def build_lora() -> LoraConfig:
    return LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        target_modules=[
            "q_proj",
            "k_proj",
            "v_proj",
            "o_proj",
            "gate_proj",
            "up_proj",
            "down_proj",
        ],
    )


def load_stories(path: Path) -> list[dict]:
    with path.open(encoding="utf-8") as fh:
        return [json.loads(line) for line in fh if line.strip()]


class StoryDataset(torch.utils.data.Dataset):
    def __init__(self, rows: list[dict], tokenizer, max_length: int):
        self.rows = rows
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self) -> int:
        return len(self.rows)

    def __getitem__(self, index: int) -> dict:
        messages = make_messages(self.rows[index]["text"])
        full = list(
            self.tokenizer.apply_chat_template(
                messages, tokenize=True, add_generation_prompt=False, return_dict=False
            )
        )[: self.max_length]
        prompt = self.tokenizer.apply_chat_template(
            messages[:1], tokenize=True, add_generation_prompt=True, return_dict=False
        )
        prompt_len = min(len(prompt), len(full))
        label = full.copy()
        # Реплика пользователя не участвует в loss: учимся только на рассказе.
        label[:prompt_len] = [-100] * prompt_len
        return {
            "input_ids": full,
            "labels": label,
            "attention_mask": [1] * len(full),
        }


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--data", type=Path, required=True)
    parser.add_argument("--out", type=Path, required=True)
    parser.add_argument("--model", default="Qwen/Qwen2.5-1.5B-Instruct")
    parser.add_argument("--epochs", type=float, default=3)
    parser.add_argument("--max-length", type=int, default=768)
    parser.add_argument("--allow-cpu", action="store_true", help="разрешить обучение без GPU (очень медленно)")
    args = parser.parse_args()

    if not torch.cuda.is_available() and not args.allow_cpu:
        raise SystemExit(
            "GPU не найден. В Colab: Runtime → Change runtime type → T4 GPU. "
            "Обучение на CPU займёт часы (для запуска всё же добавьте --allow-cpu)."
        )

    tokenizer = AutoTokenizer.from_pretrained(args.model)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(args.model, dtype=pick_dtype())
    model.config.use_cache = False
    model = get_peft_model(model, build_lora())
    # Обучаемые веса LoRA держим в float32: иначе fp16-обучение на T4
    # падает с «Attempting to unscale FP16 gradients».
    for param in model.parameters():
        if param.requires_grad:
            param.data = param.data.float()
    model.print_trainable_parameters()

    stories = load_stories(args.data)
    tokenized = StoryDataset(stories, tokenizer, args.max_length)
    collator = DataCollatorForSeq2Seq(tokenizer, pad_to_multiple_of=8, label_pad_token_id=-100)
    use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    use_fp16 = torch.cuda.is_available() and not use_bf16
    train_args = TrainingArguments(
        output_dir=str(args.out / "checkpoints"),
        num_train_epochs=args.epochs,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        learning_rate=2e-4,
        logging_steps=5,
        save_strategy="no",
        report_to="none",
        bf16=use_bf16,
        fp16=use_fp16,
        remove_unused_columns=False,
    )
    Trainer(model=model, args=train_args, train_dataset=tokenized, data_collator=collator).train()
    model.save_pretrained(args.out)
    tokenizer.save_pretrained(args.out)
    print(f"адаптер сохранён в {args.out}")


if __name__ == "__main__":
    main()

Overwriting train_lora.py


## 3. generate.py

In [7]:
%%writefile generate.py
"""Сравнение трёх прогонов одним промптом: база, адаптер Хармса, адаптер Чехова."""

from __future__ import annotations

import os

for _key, _value in list(os.environ.items()):
    if (
        _key.lower().endswith("proxy")
        and isinstance(_value, str)
        and _value.startswith("socks://")
    ):
        os.environ[_key] = "socks5://" + _value[len("socks://") :]

import argparse
from pathlib import Path

import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

BASE = "Qwen/Qwen2.5-1.5B-Instruct"

PROMPT = "Напиши рассказ о том, как человек стоит в очереди в поликлинике."

RUNS = [
    ("Базовая модель", None),
    ("LoRA: Хармс", Path("../output/harms")),
    ("LoRA: Чехов", Path("../output/chekhov")),
]


def pick_dtype() -> torch.dtype:
    if torch.cuda.is_available():
        return torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    return torch.float32


def attach_adapter(model, adapter_dir: Path):
    """Надеваем сохранённый LoRA на уже загруженную базу (без повторной загрузки)."""
    return PeftModel.from_pretrained(model, str(adapter_dir.resolve()))


def generate(
    model, tokenizer, prompt: str, temperature: float, num_predict: int
) -> str:
    # Тот же вид диалога, что в train_lora.make_messages: одна реплика user, без system.
    messages = [{"role": "user", "content": prompt}]
    input_ids = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_dict=False
    )
    input_ids = torch.tensor([input_ids], device=model.device)
    attention_mask = torch.ones_like(input_ids)
    output = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=num_predict,
        do_sample=temperature > 0,
        temperature=temperature if temperature > 0 else None,
        pad_token_id=tokenizer.pad_token_id,
    )
    new_tokens = output[0, input_ids.shape[1] :]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--model", default=BASE)
    parser.add_argument("--out", type=Path, default=Path("generations.md"))
    parser.add_argument("--temperature", type=float, default=0.8)
    parser.add_argument("--num-predict", type=int, default=400)
    parser.add_argument("--seed", type=int, default=0)
    args = parser.parse_args()

    if not PROMPT.startswith("Напиши рассказ о "):
        raise SystemExit("PROMPT должен начинаться с «Напиши рассказ о »")
    if len(RUNS) != 3:
        raise SystemExit("в RUNS нужны ровно три прогона")

    # Проверяем адаптеры заранее, до загрузки модели.
    for label, adapter in RUNS:
        if adapter is not None and not (adapter / "adapter_config.json").exists():
            raise SystemExit(
                f"Нет адаптера «{label}»: файла {adapter / 'adapter_config.json'} не существует. "
                f"Сначала обучите его: python train_lora.py --data ... --out {adapter}"
            )

    tokenizer = AutoTokenizer.from_pretrained(args.model)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(args.model, dtype=pick_dtype())
    if torch.cuda.is_available():
        model = model.cuda()
    model.eval()

    lines = [f"Промпт: {PROMPT}", ""]
    for label, adapter in RUNS:
        torch.manual_seed(args.seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(args.seed)
        current = model if adapter is None else attach_adapter(model, adapter)
        print(f"=== {label} ===", flush=True)
        text = generate(current, tokenizer, PROMPT, args.temperature, args.num_predict)
        lines += [f"### {label}", text, ""]
        if adapter is not None:
            model = current.unload()
    args.out.write_text("\n".join(lines).rstrip() + "\n", encoding="utf-8")
    print(f"записано {args.out}")


if __name__ == "__main__":
    main()

Overwriting generate.py


## 4. Подготовка данных

In [8]:
!python prepare_data.py --author harms   --out ../data/harms.jsonl
!python prepare_data.py --author chekhov --out ../data/chekhov.jsonl

качаю https://www.lib.ru/HARMS/harms.txt
harms: 118 рассказов, 238074 символов -> ../data/harms.jsonl
качаю http://az.lib.ru/c/chehow_a_p/text_0010.shtml
качаю http://az.lib.ru/c/chehow_a_p/text_0020.shtml
качаю http://az.lib.ru/c/chehow_a_p/text_0030.shtml
качаю http://az.lib.ru/c/chehow_a_p/text_0040.shtml
качаю http://az.lib.ru/c/chehow_a_p/text_0050.shtml
качаю http://az.lib.ru/c/chehow_a_p/text_0060.shtml
качаю http://az.lib.ru/c/chehow_a_p/text_0070.shtml
качаю http://az.lib.ru/c/chehow_a_p/text_0080.shtml
качаю http://az.lib.ru/c/chehow_a_p/text_0090.shtml
качаю http://az.lib.ru/c/chehow_a_p/text_0100.shtml
chekhov: 573 рассказов, 7557958 символов -> ../data/chekhov.jsonl


## 5. Обучение адаптеров

In [9]:
!python train_lora.py --data ../data/harms.jsonl   --out ../output/harms
!python train_lora.py --data ../data/chekhov.jsonl --out ../output/chekhov

# Проверка: в каждой папке должны быть adapter_config.json и adapter_model.safetensors
!ls -la ../output/harms ../output/chekhov

config.json: 100% 660/660 [00:00<00:00, 3.34MB/s]
tokenizer_config.json: 100% 7.30k/7.30k [00:00<00:00, 27.7MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 65.4MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 110MB/s]
tokenizer.json: 100% 7.03M/7.03M [00:00<00:00, 104MB/s]

model.safetensors: downloading bytes:   0% 0.00/3.09G [00:00<?, ?B/s]
model.safetensors: downloading bytes:   2% 69.4M/3.09G [00:01<00:24, 124MB/s, 3.89MB/s  ] 
model.safetensors: downloading bytes:   3% 91.6M/3.09G [00:01<00:20, 148MB/s, 6.71MB/s  ]
model.safetensors: downloading bytes:   4% 115M/3.09G [00:01<00:17, 171MB/s, 8.80MB/s  ] 
model.safetensors: downloading bytes:   5% 154M/3.09G [00:01<00:16, 177MB/s, 12.4MB/s  ]
model.safetensors: downloading bytes:   6% 187M/3.09G [00:01<00:17, 169MB/s, 15.5MB/s  ]
model.safetensors: downloading bytes:   9% 266M/3.09G [00:02<00:12, 225MB/s, 21.5MB/s  ]
model.safetensors: downloading bytes:   9% 293M/3.09G [00:02<00:11, 236MB/s, 24.3MB/s  ]
model.safetensors: download

## 6. Генерация

In [10]:
!python generate.py --out generations.md
print(open("generations.md", encoding="utf-8").read())

Loading weights: 100% 338/338 [00:00<00:00, 4442.45it/s]
=== Базовая модель ===
=== LoRA: Хармс ===
=== LoRA: Чехов ===
записано generations.md
Промпт: Напиши рассказ о том, как человек стоит в очереди в поликлинике.

### Базовая модель
Вот пример рассказа о том, как человек стоит в очереди в поликлинику:

Жил-был молодой человек по имени Иван. Он решил обратиться за медицинской помощью и отправился в больницу на улице Трудовой. Встретив там огромный стройный дежурного с блестящими бирюзовыми очками и розовыми щипцами, он понял, что очередь для приема медиков начинается уже через 10 минут.

Иван долго стоял, перебирая календари, но ни один из них не показывал нужную дату. Опять же, он прислушивался к шумам вокруг: гудки телефонов, голоса врачей и пациентов, все это добавляло ему непривычной тревоги. Наконец, когда дверь подъезда начала раскрываться, Иван, наконец, собрался и пошел внутрь. 

Он прошел мимо длинных рядов столов, где стояли люди с разнообразными проблемами. Молодой челове

In [11]:
print(open("generations.md", encoding="utf-8").read())

Промпт: Напиши рассказ о том, как человек стоит в очереди в поликлинике.

### Базовая модель
Вот пример рассказа о том, как человек стоит в очереди в поликлинику:

Жил-был молодой человек по имени Иван. Он решил обратиться за медицинской помощью и отправился в больницу на улице Трудовой. Встретив там огромный стройный дежурного с блестящими бирюзовыми очками и розовыми щипцами, он понял, что очередь для приема медиков начинается уже через 10 минут.

Иван долго стоял, перебирая календари, но ни один из них не показывал нужную дату. Опять же, он прислушивался к шумам вокруг: гудки телефонов, голоса врачей и пациентов, все это добавляло ему непривычной тревоги. Наконец, когда дверь подъезда начала раскрываться, Иван, наконец, собрался и пошел внутрь. 

Он прошел мимо длинных рядов столов, где стояли люди с разнообразными проблемами. Молодой человек заметил, что некоторые из них пили кофе, другие — поглощали конфеты, а еще кто-то просто смотрел телевизор. 

Пока Иван стоял, его внимание бы

In [12]:
from google.colab import drive
drive.flush_and_unmount()

## 7. Разбор

База отвечает как ассистент: «Вот пример рассказа о том, как человек стоит в очереди в поликлинику:» и дальше сказочный зачин «Жил-был молодой человек по имени Иван». Текст описательный.

Адаптер Хармса убрал вводную фразу и перевёл текст в короткие реплики:
«- Садитесь! - кричит женщина…», «Женщина протягивает ему чистую бумагу: - Пишите: "Я не могу прийти"». Нет внезапного обрыва или абсурдной развязки, фраза «Женщина протягивает…» просто повторяется. Лексика случайная и современная, а сюжет про очередь забыт.

Адаптер Чехова тоже убрал вводную фразу и начал с заголовка: заголовок «ПОЛИКЛИНИКА», диалог через «--», рассказ от первого лица. Есть интонация перечисления: «Сколько я ее прохожу, сколько людей в нее входит и выходят, сколько раз меня обгоняют». Сюжет уходит к «кукурузному крахмалу» и «Разгулу розовых мух», выдуманы слова «прелестица», «жуличье пюре». Адаптер выучил и мусор из данных: «&nbsp;&nbsp;» и жёсткие переносы, потому что при чистке не заменена сущность &nbsp; и не убраны символы \r.

Итог: оба адаптера изменили форму (убрали тон ассистента, задали формат диалога и ритм), но не содержание и не язык: темы теряются, а лексику 1.5B-модель выдумывает. Вывод основан на одном сэмпле на модель, тексты обрезаны по лимиту токенов.